# Apply v2e temporal offset (one-time data prep)

`config.yaml` defines a `v2e_temporal_offset` (currently 19000 us) needed to align the
v2e-simulated event stream with the real event stream in a common timeline. The
analysis pipeline (`main.py`) used to apply this offset at runtime by mutating the
loaded event data in place, but that required opening the huge source `.h5` file in
write mode and re-applied the offset cumulatively on every run (a real bug).

Instead, this notebook applies the offset **once**, streaming the source dataset in
chunks (never loading the full multi-million-event array into memory), and writes a
new `.h5` file with `t` shifted. The original source file is never opened for writing
and is left untouched.

After running this once, point `v2e_data_path` in `config.yaml` at the generated
output file and set `v2e_temporal_offset: 0`.


In [1]:
# Initialization
from __future__ import annotations

from pathlib import Path

import h5py
import numpy as np

# Source v2e file (read-only, never modified).
SRC_PATH = Path(r"C:/Users/cxm3593/Academic/Workspace/Data/data_sync/v2e_event_undistorted_normalized.h5")

# Destination file: offset-applied copy, written alongside the source.
DST_PATH = SRC_PATH.with_name(SRC_PATH.stem + "_offset19000.h5")

DATASET_NAME = "events"
TIME_FIELD = "t"
OFFSET_US = 19000

# Rows processed per chunk; keeps peak memory small regardless of file size.
CHUNK_ROWS = 1_000_000


In [2]:
def apply_offset_to_h5(
    src_path: Path,
    dst_path: Path,
    offset: int,
    dataset_name: str = DATASET_NAME,
    time_field: str = TIME_FIELD,
    chunk_rows: int = CHUNK_ROWS,
    overwrite: bool = False,
) -> Path:
    """Write a copy of ``src_path`` with ``time_field`` shifted by ``offset``.

    Streams the dataset in ``chunk_rows``-sized slices so peak memory stays
    bounded regardless of the source file's total event count. The source
    file is opened read-only and is never modified.
    """
    if dst_path.exists() and not overwrite:
        raise FileExistsError(f"{dst_path} already exists (pass overwrite=True to replace it).")

    with h5py.File(src_path, "r") as src_file:
        src_dataset = src_file[dataset_name]
        n_rows = src_dataset.shape[0]
        dtype = src_dataset.dtype

        if time_field not in dtype.names:
            raise KeyError(f"{time_field!r} not found in dataset fields: {dtype.names}")

        with h5py.File(dst_path, "w") as dst_file:
            dst_dataset = dst_file.create_dataset(
                dataset_name,
                shape=src_dataset.shape,
                dtype=dtype,
                chunks=src_dataset.chunks,
                compression=src_dataset.compression,
                compression_opts=src_dataset.compression_opts,
            )
            dst_file.attrs.update(src_file.attrs)
            dst_dataset.attrs.update(src_dataset.attrs)

            time_dtype = dtype[time_field]
            n_chunks = (n_rows + chunk_rows - 1) // chunk_rows
            for chunk_index, start in enumerate(range(0, n_rows, chunk_rows)):
                end = min(start + chunk_rows, n_rows)

                chunk = src_dataset[start:end]  # fresh in-memory copy; safe to mutate
                shifted_time = chunk[time_field].astype(np.int64) + offset

                info = np.iinfo(time_dtype)
                if shifted_time.max() > info.max or shifted_time.min() < info.min:
                    raise OverflowError(
                        f"Offset shifted {time_field!r} outside {time_dtype} range "
                        f"at rows [{start}:{end}]."
                    )

                chunk[time_field] = shifted_time.astype(time_dtype)
                dst_dataset[start:end] = chunk

                if chunk_index % 10 == 0 or end == n_rows:
                    print(f"  wrote rows {start:,} - {end:,} / {n_rows:,} (chunk {chunk_index + 1}/{n_chunks})")

    print(f"Done: {dst_path}")
    return dst_path


In [3]:
apply_offset_to_h5(SRC_PATH, DST_PATH, OFFSET_US, overwrite=False)


  wrote rows 0 - 1,000,000 / 55,978,835 (chunk 1/56)
  wrote rows 10,000,000 - 11,000,000 / 55,978,835 (chunk 11/56)
  wrote rows 20,000,000 - 21,000,000 / 55,978,835 (chunk 21/56)
  wrote rows 30,000,000 - 31,000,000 / 55,978,835 (chunk 31/56)
  wrote rows 40,000,000 - 41,000,000 / 55,978,835 (chunk 41/56)
  wrote rows 50,000,000 - 51,000,000 / 55,978,835 (chunk 51/56)
  wrote rows 55,000,000 - 55,978,835 / 55,978,835 (chunk 56/56)
Done: C:\Users\cxm3593\Academic\Workspace\Data\data_sync\v2e_event_undistorted_normalized_offset19000.h5


WindowsPath('C:/Users/cxm3593/Academic/Workspace/Data/data_sync/v2e_event_undistorted_normalized_offset19000.h5')

In [4]:
# Sanity check: compare a few rows between source and generated file.
with h5py.File(SRC_PATH, "r") as f_src, h5py.File(DST_PATH, "r") as f_dst:
    src_t = f_src[DATASET_NAME][:5][TIME_FIELD]
    dst_t = f_dst[DATASET_NAME][:5][TIME_FIELD]
    print("source t     :", src_t)
    print("offset t     :", dst_t)
    print("expected diff:", OFFSET_US)
    assert np.all(dst_t.astype(np.int64) - src_t.astype(np.int64) == OFFSET_US)
    print("OK: offset applied correctly, source file unchanged.")


source t     : [0 0 0 0 0]
offset t     : [19000 19000 19000 19000 19000]
expected diff: 19000
OK: offset applied correctly, source file unchanged.
